# 시간 설정

In [1]:
import time
import random
import os

os.environ['TZ'] = 'Asia/Seoul'
time.tzset()

random.seed(1490)

def print_time() -> None:
    print(time.strftime('%Y년 %m월 %d일 %A %p %I시 %M분 %S초'))
    return

def timestamp() -> str:
    return time.strftime('%Y-%m-%d-%H-%M-%S')

print_time()
print(timestamp())

2024년 08월 24일 Saturday AM 12시 30분 55초


# 제품 이상여부 판별 프로젝트


## 필수 라이브러리


In [3]:
import os
from pprint import pprint

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn import metrics
from sklearn.model_selection import train_test_split
from tqdm import tqdm

from sklearn.preprocessing import LabelEncoder

import matplotlib.pyplot as plt  # 그래프 보기
import seaborn as sns  # 그래프 보기

print_time()

2024년 08월 24일 Saturday AM 12시 30분 57초


## 전역변수

In [4]:
global drops

# ROOT_PREFIX = "/content/drive/MyDrive/lgaimers5/" # 구글 드라이브용 루트
# ROOT_DIR = ROOT_PREFIX + "data" # 에이머스 제출 시 주석 처리 필요
ROOT_DIR = "data" # 구글 드라이브 실행 시 주석 처리 필요

RANDOM_STATE = 110 # 랜덤 시드
corr_thresh = 0.0001 # 상관관계 삭제 기준
empty_thresh = 30 # 결측값 삭제 기준 비율

time_lim = None # 유전 알고리즘 실행 제한 시간(초)
popula = 25 # 유전 알고리즘 세대 별 개체 수
ngen = 20 # 유전 알고리즘 세대 수

# 랜덤 시드 설정
np.random.seed(RANDOM_STATE)

# 1. 필터 1번, 필터 2번 등과 같이 순서가 있는 값은 추가 처리 없이 넘버링,
# 그 외 기타 order들도 같이 넘버링
numbering = ['Equipment_Dam', 'Model.Suffix_Dam', 'Workorder_Dam', \
             'Model.Suffix_AutoClave', 'Workorder_AutoClave', 'Equipment_Fill1', \
             'Model.Suffix_Fill1', 'Workorder_Fill1', 'Equipment_Fill2', \
             'Model.Suffix_Fill2', 'Workorder_Fill2']

# 2. OK, NG와 같이 범주형인 경우 가능하면 원 핫 인코딩
onehot = ['HEAD NORMAL COORDINATE X AXIS(Stage1) Judge Value_Dam', \
          'GMES_ORIGIN_INSP_JUDGE_CODE Collect Result_AutoClave', \
          'GMES_ORIGIN_INSP_JUDGE_CODE Judge Value_AutoClave', \
          'HEAD NORMAL COORDINATE X AXIS(Stage1) Judge Value_Fill1', \
          'HEAD NORMAL COORDINATE X AXIS(Stage1) Judge Value_Fill2', \
          'Chamber Temp. Judge Value_AutoClave']

# 3. 숫자 값이 들어가야 하는데 OK가 혼입된 값임
custom = ['HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Dam', \
          'HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill1', \
          'HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill2']

drops = [] # 값이 없거나 무의미해서 버릴 column들
dt_nan = []  # attributes to fill the missing value with any value
dt_del = []  # attributes to be deleted because there are too many missing values
need_general = []  # normalization list
del_list = [] # 상관관계 삭제 리스트
reduce_list = [] # 다중공선성 삭제 리스트
# 다중공선성: 서로 다른 속성끼리 깊은 연관을 갖는 것.
# 예를 들면 타겟은 따로 있는데 A 속성과 B 속성 둘 사이의 상관관계 값의 절댓값이 1에 가깝게 나오는 경우를 말함

print_time()

2024년 08월 24일 Saturday AM 12시 30분 57초


## 함수 정의

In [5]:
def print_col_uniqs(data):
    global drops
    """
    unique 개수가 1개 이하인 column 삭제
    """
    init_cols = len(data.columns)
    total_cnt = len(data)

    for col in data.columns:
        if col == 'target':
            continue
        uniqs = data[col].unique()
        nuq = data[col].nunique()
        vcnt = data[col].value_counts()
        if (nuq <= 1 and len(uniqs) <= 1) or (len(data) == data[col].isna().sum()):
            drops.append(col)
        else:
            if nuq <= 10:
                # print(f'{col} : {nuq}, {uniqs}')
                pass
            else:
                # print(f'{col} : {nuq}')
                pass
            # 결측값 비율 보기(null이 있는 속성만)
            cnt = data[col].isna().sum()

            if cnt == 0:
                continue

            num = float(cnt) / float(total_cnt)

            if num * 100.0 <= empty_thresh:
                dt_nan.append(col)
            else:
                dt_del.append(col)
    
    drops += dt_del
    print(f'drops: {len(drops)}\ndt_del: {len(dt_del)}')
    data.drop(drops, axis=1, inplace=True)
    data.drop_duplicates(keep='first', inplace=True, ignore_index=True) # 중복 제거
    data[list(data.columns)].dropna(axis=0, inplace=True) # 모든 속성이 결측값인 행 제거
    
    # data.fillna(-1, inplace = True)
    for col in data.columns:
        if str(data[col].dtypes) != 'object':
            data[col] = data[col].fillna(data[col].mean())
        else:
            data[col].fillna(-1, inplace = True)
    print(f'결측값 처리 후 결측값 개수: {data.isna().sum().sum()}')
    print(f'num of initial cols: {init_cols}')
    print(f'num of deleted cols: {len(drops)}')
    print(f'num of remained cols: {len(data.columns)}')
    return data, drops

def print_columns_to_list(data):
    """
    컬럼이 너무 많아 바로 출력이 안되니 직접 출력
    """
    cnt = {}
    for col in data.columns:
        dt = str(data[col].dtypes)
        nu = data[col].nunique()
        if dt in cnt.keys():
            cnt[dt] += 1
        else:
            cnt[dt] = 1
        print(f'* {col} : {dt}({nu})')
    print(f'\ninfo: {cnt}')

def object_unique(data):
    """
    dtype이 object로 분류된 컬럼의 unique 값 확인
    """
    idx = 1
    for col in data.columns:
        if str(data[col].dtypes) == 'object':
            print(f'{idx}. {col}\n\t{data[col].nunique()}', end='')
            if data[col].nunique() <= 20:
                print(f', {data[col].unique()}')
            else:
                print()
            idx += 1

def enc_numbering(data, cols):
    """
    단순 넘버링으로 라벨 인코딩
    """
    print('넘버링')
    for col in cols:
        if col not in data.columns:
            continue
        data[col] = data[col].astype(str)
        le = LabelEncoder()
        le.fit(sorted(list(data[col].unique())))
        data[col] = le.transform(data[col])
    return data

def enc_onehot(data, cols):
    """
    원 핫 인코딩
    """
    print('원 핫 인코딩')
    for c in cols:
        if c not in data.columns:
            cols.remove(c)
    data = pd.get_dummies(data, columns = cols)
    return data

def enc_only_obj(data, cols):
    """
    다른 컬럼의 OK가 혼입된 값 정리, 소수 값을 리터럴 소수로 변환 후 dtype도 소수로 지정
    """
    print('문자열 숫자를 숫자로')
    for col in cols:
        if col not in data.columns:
            continue
        idx = 1
        uniqs = data[col].unique()
        for u in uniqs:
            try:
                data.loc[data[col] == u, col] = float(u)
            except:
                if u == 'OK' or u == -1:
                    data.loc[data[col] == u, col] = -1
                else:
                    data.loc[data[col] == u, col] = idx
                    idx += 1
        data[col] = data[col].astype(float)
        # print(data[col])
    return data

def obj_encoding(data, mode = 'train'):
    """
    모든 라벨 인코딩 실행
    """
    # 기본 넘버링
    data = enc_numbering(data, numbering)
    # 원 핫 인코딩
    # data = enc_onehot(data, onehot)
    data = enc_numbering(data, onehot)
    # OK가 혼입된 값 정리
    data = enc_only_obj(data, custom)
    # target 인코딩
    # 있는 경우에만
    if mode == 'train':
        print('target 값을 숫자로')
        data.loc[data['target'] == 'AbNormal', 'target'] = 1
        data.loc[data['target'] == 'Normal', 'target'] = 0
    return data

def show_heatmap(data, cnt = 12):
    """
    컬럼을 지정된 수만큼 묶어서 히트맵 출력
    * 색깔 참고: https://colorbrewer2.org/
    """
    cols = list(data.columns)
    corr = data.corr()
    print(corr.info())

    # plt.figure(figsize=(15, 10))
    # sns.heatmap(data.corr(), annot=True, cmap="RdBu", vmax=1)
    # plt.show()

def del_by_corr(data, thresh = 0.5, del_list = []):
    """
    지정된 값을 기준으로 상관관계가 낮은 컬럼 삭제
    """
    corr = data.corr()
    target_corr = corr['target']
    target_corr.drop('target', inplace=True)
    del_list = list(target_corr[target_corr.isnull() == True].index)
    temp = target_corr[-thresh < target_corr]
    del_list += list(temp[temp < thresh].index)

    data.drop(del_list, axis=1, inplace=True)
    print(f'num of cols deleted by correlation: {len(del_list)}')
    print(f'remain cols: {len(data.columns)}')

    return data, del_list

def corr_reduce(data, thresh=0.9, reduce_list=reduce_list):
    # 상관계수 행렬 계산
    correlation_matrix = data.corr().abs()

    # 상관관계 계수가 threshold 이상인 피처들 중 하나를 제거하기 위한 마스크 생성
    upper_triangle = correlation_matrix.where(
        np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)
    )

    # target_column과 연관된 피처를 제외하기 위해, target_column과 관련된 피처를 식별
    related_to_target = correlation_matrix['target'].index

    # 제거할 피처 리스트 생성
    to_drop = [column for column in upper_triangle.columns
               if column != 'target' and any(upper_triangle[column] > thresh)]

    # 상관관계가 높은 피처 제거
    df_reduced = data.drop(columns=to_drop)

    print("제거된 속성 수:", len(to_drop))

    return df_reduced, to_drop

def pre_processing(data, del_list=del_list, reduce_list=reduce_list):
    """
    데이터 전처리 통합 실행
    """
    # 데이터 뒤섞기
    print(f'# 데이터 뒤섞기')
    data = data.sample(frac=1)  # randomize data

    # unique 개수가 1개 이하인 column 삭제
    print('# unique 개수가 1개 이하인 column 삭제')
    data, drops = print_col_uniqs(data)

    # 모든 라벨 인코딩 실행
    print('# 모든 라벨 인코딩 실행')
    data = obj_encoding(data, 'train')

    # 상관관계가 낮은 컬럼 삭제
    print('# 상관관계가 낮은 컬럼 삭제')
    data, del_list = del_by_corr(data, thresh=corr_thresh, del_list=del_list)
    
    # 다중공선성이 높은 컬럼 삭제
    print('# 다중공선성이 높은 컬럼 삭제')
    data, reduce_list = corr_reduce(data)

    print('target 값 정수화')
    data['target'] = data['target'].astype(int)

    # 전처리 완료
    print('# 전처리 완료')
    print_time()

    return data, drops, del_list, reduce_list

def pre_processing_test(data):
    """
    테스트 데이터(레이블 X) 전처리 통합 실행
    """
    # unique 개수가 1개 이하인 column 삭제
    print('# unique 개수가 1개 이하인 column 삭제')
    print('속성 삭제')
    data.drop(drops, axis=1, inplace=True)
    for col in data.columns:
        if str(data[col].dtypes) != 'object':
            data[col] = data[col].fillna(data[col].mean())
        else:
            data[col].fillna(-1, inplace = True)
    print(f'결측값 처리 후 결측값 개수: {data.isna().sum().sum() - data["target"].isna().sum()}')

    # 모든 라벨 인코딩 실행
    print('# 모든 라벨 인코딩 실행')
    data = obj_encoding(data, 'test')

    # 상관관계가 낮은 컬럼 삭제
    print('# 상관관계가 낮은 컬럼 삭제')
    print(f'num of cols deleted by correlation: {len(del_list)}')
    data.drop(del_list, axis=1, inplace=True)
    print(f'remain cols: {len(data.columns)}')
    
    # 다중공선성이 높은 컬럼 삭제
    print('# 다중공선성이 높은 컬럼 삭제')
    data = data.drop(columns=reduce_list)

    # 테스트 데이터 전처리 완료
    print('# 테스트 데이터 전처리 완료')
    print_time()

    return data

print_time()

2024년 08월 24일 Saturday AM 12시 30분 57초


## 1. 데이터 불러오기


### 데이터 읽어오기


In [6]:
print_time()

# Load data
train_data = pd.read_csv(os.path.join(ROOT_DIR, "train.csv"))

# 데이터 전처리
train_data, drops, del_list, reduce_list = pre_processing(train_data)

2024년 08월 24일 Saturday AM 12시 30분 57초
# 데이터 뒤섞기
# unique 개수가 1개 이하인 column 삭제
drops: 321
dt_del: 8
결측값 처리 후 결측값 개수: 0
num of initial cols: 464
num of deleted cols: 321
num of remained cols: 143
# 모든 라벨 인코딩 실행
넘버링
넘버링
문자열 숫자를 숫자로
target 값을 숫자로
# 상관관계가 낮은 컬럼 삭제
num of cols deleted by correlation: 0
remain cols: 143
target 값 정수화
전처리 완료


In [7]:
print(f'drops: {len(drops)}, del_list: {len(del_list)}, reduce_list: {len(reduce_list)}')

321 0


### columns info

In [8]:
# 전체 컬럼 정보 출력
print_columns_to_list(train_data)

* Equipment_Dam : int64(2)
* Model.Suffix_Dam : int64(7)
* Workorder_Dam : int64(663)
* CURE END POSITION X Collect Result_Dam : float64(2)
* CURE END POSITION Z Collect Result_Dam : float64(2)
* CURE END POSITION Θ Collect Result_Dam : int64(2)
* CURE SPEED Collect Result_Dam : int64(5)
* CURE START POSITION X Collect Result_Dam : int64(2)
* CURE START POSITION Θ Collect Result_Dam : int64(2)
* DISCHARGED SPEED OF RESIN Collect Result_Dam : int64(3)
* DISCHARGED TIME OF RESIN(Stage1) Collect Result_Dam : float64(19)
* DISCHARGED TIME OF RESIN(Stage2) Collect Result_Dam : float64(29)
* DISCHARGED TIME OF RESIN(Stage3) Collect Result_Dam : float64(20)
* Dispense Volume(Stage1) Collect Result_Dam : float64(23)
* Dispense Volume(Stage2) Collect Result_Dam : float64(33)
* Dispense Volume(Stage3) Collect Result_Dam : float64(22)
* HEAD NORMAL COORDINATE X AXIS(Stage2) Collect Result_Dam : float64(41)
* HEAD NORMAL COORDINATE X AXIS(Stage3) Collect Result_Dam : float64(40)
* HEAD NORMAL COOR

### object uniques

In [9]:
# object 데이터의 unique 값 확인
object_unique(train_data)

print_time()

2024년 08월 24일 Saturday AM 12시 31분 01초


In [10]:
# info 확인
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40506 entries, 0 to 40505
Columns: 143 entries, Equipment_Dam to target
dtypes: float64(68), int64(75)
memory usage: 44.2 MB


In [11]:
train_data.isna().sum().sum()

0

In [12]:
# 컬럼 리스트 확인
# 리스트로 변환하면 생략하지 않고 출력됨
# list(train_data.columns)

### ADASYN 오버 샘플링

https://dapin1490.github.io/satinbower/posts/it-deeplearning-data-2/#3-adasyn-%EC%98%A4%EB%B2%84-%EC%83%98%ED%94%8C%EB%A7%81

In [13]:
!pip install imblearn

Defaulting to user installation because normal site-packages is not writeable


In [14]:
import imblearn  # 샘플링 관련 라이브러리
print("\nimblearn 버전 확인\n{}".format(imblearn.__version__))

# ADASYN 오버 샘플링
# 참고 https://dining-developer.tistory.com/27
from imblearn.over_sampling import ADASYN

temp_x = train_data.drop('target', axis=1)
temp_y = train_data['target']

temp_x, temp_y = ADASYN(sampling_strategy=0.3).fit_resample(temp_x, temp_y)

df_concat = pd.concat([temp_x, temp_y], axis=1).reset_index(drop=True)
df_concat.value_counts('target')


imblearn 버전 확인
0.12.3


target
0    38156
1    11057
Name: count, dtype: int64

### 언더 샘플링


데이타 불균형을 해결하기 위해 언더 샘플링을 진행합니다.


In [15]:
normal_ratio = 3.2  # 1.0 means 1:1 ratio

df_normal = df_concat[df_concat["target"] == 0]
df_abnormal = df_concat[df_concat["target"] == 1]

num_normal = len(df_normal)
num_abnormal = len(df_abnormal)
print(f"Total: Normal: {num_normal}, AbNormal: {num_abnormal}")

df_normal = df_normal.sample(n=int(num_abnormal * normal_ratio), replace=False, random_state=RANDOM_STATE)
df_concat = pd.concat([df_normal, df_abnormal], axis=0).reset_index(drop=True)
print(df_concat.value_counts("target"))

print()
print_time()

Total: Normal: 38156, AbNormal: 11057
target
0    13268
1    11057
Name: count, dtype: int64

2024년 08월 24일 Saturday AM 12시 31분 04초


### 데이터 분할


In [16]:
print_time()
print()

df_train, df_val = train_test_split(
    df_concat,
    test_size=0.2,
    stratify=df_concat["target"],
    random_state=RANDOM_STATE,
)

def print_stats(df: pd.DataFrame):
    num_normal = len(df[df["target"] == 0])
    num_abnormal = len(df[df["target"] == 1])

    print(f"  Total: Normal: {num_normal}, AbNormal: {num_abnormal}" + f" ratio: {num_abnormal / num_normal}")

# Print statistics
print(f"  \tNormal    \tAbNormal")
print_stats(df_train)
print_stats(df_val)

2024년 08월 24일 Saturday AM 12시 31분 04초

  	Normal    	AbNormal
  Total: Normal: 10614, AbNormal: 8846 ratio: 0.8334275485208216
  Total: Normal: 2654, AbNormal: 2211 ratio: 0.8330821401657875


In [17]:
features = []

for col in df_train.columns:
    try:
        # df_train[col] =
        df_train[col].astype(int) # float를 그대로 두기 위해 일부러 줄을 내렸으나, object 컬럼을 제외한다는 것은 유지하기 위해 이 부분은 놔둠
        features.append(col)
    except:
        continue

if 'target' in features:
    features.remove('target')

train_x = df_train[features]
train_y = df_train["target"]

print_time()
train_x.info()

2024년 08월 24일 Saturday AM 12시 31분 04초
<class 'pandas.core.frame.DataFrame'>
Index: 19460 entries, 3113 to 11694
Columns: 142 entries, Equipment_Dam to WorkMode Collect Result_Fill2
dtypes: float64(68), int64(74)
memory usage: 21.2 MB


In [18]:
# 학습 데이터 기준으로 f1 score 채점
test_x = df_val[features]
test_y = df_val["target"]
pred_y = []

print_time()

2024년 08월 24일 Saturday AM 12시 31분 04초


테스트 데이터 불러오기

In [ ]:
test_data = pd.read_csv(os.path.join(ROOT_DIR, "test.csv"))
test_data.drop('Set ID', axis=1, inplace=True)
test_data = pre_processing_test(test_data)

print_time()
test_data.info()

## 유전 알고리즘 속성 선택

유전 알고리즘을 이용해 속성을 선택할 때, 원본 데이터를 사용할지 오버샘플링된 데이터를 사용할지에 대한 결정은 다음과 같은 여러 요소를 고려해야 합니다.

### 1. 오버샘플링의 목적
오버샘플링은 일반적으로 데이터 불균형 문제를 해결하기 위해 사용됩니다. 소수 클래스의 데이터를 증강하여 모델이 소수 클래스를 무시하지 않고 잘 학습할 수 있도록 하는 것이 주된 목적입니다. 따라서, 오버샘플링이 중요한 경우에는 최종 모델 학습에 오버샘플링된 데이터를 사용하는 것이 좋습니다.

### 2. 속성 선택의 목표
유전 알고리즘을 사용해 속성을 선택할 때의 목표는 모델 성능을 극대화하는 속성 집합을 찾는 것입니다. 따라서, 속성 선택에 사용하는 데이터는 최종 모델이 학습할 데이터와 일치해야 합니다.

* 원본 데이터 사용: 원본 데이터를 사용하면 속성 선택 과정에서 모델이 오버샘플링으로 인한 데이터 불균형 문제를 고려하지 않게 됩니다. 이는 원본 데이터에서 중요한 속성을 찾을 수는 있지만, 오버샘플링된 데이터를 사용하는 경우와 일관성이 떨어질 수 있습니다.
* 오버샘플링된 데이터 사용: 오버샘플링된 데이터를 사용하면, 속성 선택 과정이 최종적으로 사용될 데이터 분포를 반영하게 됩니다. 이는 오버샘플링된 데이터에 대해 모델 성능을 최적화하는 데 도움이 될 수 있습니다. 다만, 오버샘플링으로 인해 데이터에 추가된 노이즈나 중복된 정보가 속성 선택에 영향을 미칠 수 있다는 단점이 있습니다.

### 3. 실험적 접근
이론적으로는 오버샘플링된 데이터를 사용하는 것이 일관성을 보장하는 좋은 방법이지만, 실제로는 다음과 같은 접근을 권장합니다:

1. 원본 데이터와 오버샘플링된 데이터를 각각 사용해 속성 선택: 두 가지 데이터를 사용해 유전 알고리즘을 돌리고, 각각의 경우에서 선택된 속성을 비교합니다. 이 방법을 통해 오버샘플링이 속성 선택에 미치는 영향을 평가할 수 있습니다.
2. 모델 성능 평가: 각각의 속성 집합을 사용해 모델을 학습시키고, 검증 데이터(오버샘플링되지 않은 데이터 또는 별도로 확보한 테스트 데이터)에서 성능을 평가합니다. 이를 통해 어떤 데이터로 속성을 선택했을 때 최종 성능이 더 좋은지 판단할 수 있습니다.
3. 하이브리드 접근: 원본 데이터와 오버샘플링된 데이터를 혼합하여 유전 알고리즘을 수행할 수도 있습니다. 예를 들어, 초기 세대는 원본 데이터로 평가하고, 이후 세대는 오버샘플링된 데이터를 사용하는 방식으로 접근할 수 있습니다.

### 결론
속성 선택에 사용하는 데이터는 최종 모델 학습에 사용하는 데이터와 일관성이 있어야 하므로, 오버샘플링된 데이터를 사용하는 것이 일반적으로 권장됩니다. 다만, 각 접근법의 결과를 비교하여 최적의 방법을 결정하는 것이 중요합니다.

In [19]:
!pip install deap

Defaulting to user installation because normal site-packages is not writeable


In [20]:
import numpy as np
import random
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier
from deap import base, creator, tools, algorithms
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import VotingClassifier

def cal_range(data):
    return np.max(data) - np.min(data)

# 개체 평가
def eval_individual(individual):
    """
    * 개체는 이진 벡터로 표현되며, 각 요소는 특정 특징이 선택되었는지 여부를 나타냅니다.
    * 선택된 특징들로 모델을 학습하고, 교차 검증을 통해 성능을 평가합니다.
    """
    selected_features = [index for index, value in enumerate(individual) if value == 1]
    if len(selected_features) == 0:
        return [0]
    
    knns = [
        KNeighborsClassifier(n_neighbors=12, weights='distance', p=1),
        KNeighborsClassifier(n_neighbors=20, weights='distance', p=1),
        KNeighborsClassifier(n_neighbors=2, weights='distance', p=1)
    ]
    
    X_selected = X.iloc[:, selected_features]
    classifier = VotingClassifier(estimators=[('KNN1', knns[0]), ('KNN2', knns[1]), ('KNN3', knns[2])], voting='soft')
    scores = cross_val_score(classifier, X_selected, y, cv=5, scoring='f1')
    """
    cross_val_score()는 Scikit-learn 라이브러리에서 제공하는 함수로,
    교차 검증(cross-validation)을 통해 모델의 성능을 평가하는 데 사용됩니다.
    이 함수는 주어진 데이터셋에서 모델을 여러 번 학습시키고 평가하여,
    모델의 일반화 성능(즉, 새로운 데이터에 대한 성능)을 추정합니다.
    
    * estimator: 학습할 모델 객체 (예: RandomForestClassifier, SVC 등).
    * cv: 교차 검증에서 사용할 폴드의 수. 기본값은 5입니다.
    * scoring: 평가 지표를 지정합니다. (예: 'accuracy', 'f1', 'roc_auc' 등)
    
    위의 유전 알고리즘 코드에서는 cross_val_score()가
    각 개체(특정 특징 조합)에 대한 모델의 성능을 평가하는 데 사용됩니다.
    선택된 특징들로 모델을 학습하고, 교차 검증을 통해
    그 특징 조합이 얼마나 좋은지(즉, 모델의 성능이 얼마나 뛰어난지)를 평가하는 것입니다.
    이 평가 결과가 유전 알고리즘의 적합도(fitness) 값으로 사용됩니다.
    """
    return [scores.mean()]

# 유전 알고리즘 실행
def start_gene(tilim = None):
    """
    * eaSimple 함수를 사용하여 유전 알고리즘을 실행합니다.
      이 과정에서 교차(cxpb), 돌연변이(mutpb), 그리고 세대(ngen) 등 여러 매개변수를 지정합니다.
    * HallOfFame을 사용해 최고의 개체를 저장합니다.
    """
    random.seed(RANDOM_STATE)
    pop = toolbox.population(n=popula)
    hof = tools.HallOfFame(1)
    logbook = None
    
    # 통계 수집
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("avg", np.mean)
    stats.register("std", np.std)
    stats.register("min", np.min)
    stats.register("max", np.max)
    stats.register("range", cal_range)
    
    # 시간 제한 설정
    if tilim is None:
        print('there is no time limit')
        print('start 20 generation')
        start_time = time.time()
        pop, logbook = algorithms.eaSimple(pop, toolbox, cxpb=0.7, mutpb=0.3, ngen=ngen, 
                                           stats=stats, halloffame=hof, verbose=True)
        time_now = time.time()
        print(f'time passed {int(time_now - start_time) // 60} min {int(time_now - start_time) % 60} seconds\n')
        return pop, logbook, hof
    
    print(f'time limit {tilim // 60} min {tilim % 60} seconds')
    start_time = time.time()

    # 유전 알고리즘 실행
    cnt = 1
    for gen in range(ngen):
        # 시간 제한 확인
        time_now = time.time()
        print(f'time passed {int(time_now - start_time) // 60} min {int(time_now - start_time) % 60} seconds\n')
        if tilim is not None and time_now - start_time > tilim:
            print(f"Time limit of {tilim // 60} min {tilim % 60} seconds reached.")
            break
        
        # 다음 세대 생성
        print(f'gen # {cnt} created')
        pop, logbook = algorithms.eaSimple(pop, toolbox, cxpb=0.7, mutpb=0.3, ngen=1, 
                                           stats=stats, halloffame=hof, verbose=True)
        cnt += 1
    
    return pop, logbook, hof

print_time()

2024년 08월 24일 Saturday AM 12시 31분 05초


In [21]:
# 데이터 로드
X = train_x
y = train_y
n_features = X.shape[1]

# 유전 알고리즘 설정
# creator를 사용해 적합도(FitnessMax)와 개체(Individual) 클래스를 생성합니다.
print(f'적합도, 개체 생성')
if not hasattr(creator, "FitnessMax"):
    creator.create("FitnessMax", base.Fitness, weights=(1.0,))
    print('* 적합도 클래스 생성됨')
if not hasattr(creator, "Individual"):
    creator.create("Individual", list, fitness=creator.FitnessMax)
    print('* 개체 클래스 생성됨')

# toolbox에서 유전 알고리즘에 필요한 함수들을 정의합니다.
# 여기에는 개체 초기화, 평가, 교차, 돌연변이, 선택 함수가 포함됩니다.
print(f'유전 알고리즘 함수 정의')
toolbox = base.Toolbox()
toolbox.register("attr_bool", random.randint, 0, 1)
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_bool, n=n_features)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

toolbox.register("evaluate", eval_individual)
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutFlipBit, indpb=0.05)
toolbox.register("select", tools.selTournament, tournsize=3)

print(f'유전 알고리즘 실행')
pop, log, hof = start_gene(tilim=time_lim) # tilim: 시간 제한(초)
best_individual = hof[0]
selected_features = [index for index, value in enumerate(best_individual) if value == 1]

적합도, 개체 생성
* 적합도 클래스 생성됨
* 개체 클래스 생성됨
유전 알고리즘 함수 정의
유전 알고리즘 실행
time limit 30 min 0 seconds
time passed 0.00 seconds

gen # 1 created
gen	nevals	avg     	std      	min     	max    
0  	50    	0.757767	0.0194223	0.683403	0.77972
1  	31    	0.768444	0.0125471	0.70004 	0.78117
time passed 605.81 seconds

gen # 2 created
gen	nevals	avg     	std      	min    	max    
0  	0     	0.768444	0.0125471	0.70004	0.78117
1  	22    	0.775421	0.00351337	0.763539	0.781385
time passed 774.51 seconds

gen # 3 created
gen	nevals	avg     	std       	min     	max     
0  	0     	0.775421	0.00351337	0.763539	0.781385
1  	35    	0.774145	0.00951686	0.745731	0.78184 
time passed 1043.67 seconds

gen # 4 created
gen	nevals	avg     	std       	min     	max    
0  	0     	0.774145	0.00951686	0.745731	0.78184
1  	32    	0.775747	0.00866163	0.740127	0.782343
time passed 1294.55 seconds

gen # 5 created
gen	nevals	avg     	std       	min     	max     
0  	0     	0.775747	0.00866163	0.740127	0.782343
1  	27    	0.7792

KeyboardInterrupt: 

In [ ]:
# 최적의 특징 조합을 출력합니다.
# print("Best Individual: ", best_individual)
# print("Selected Features: ", selected_features)
print(f"Selected Features ({len(selected_features)})\n***************************")
cols = list(train_x.columns)
for i in selected_features:
    print(cols[i])

In [ ]:
features = []
cols = list(train_x.columns)
for i in selected_features:
    features.append(cols[i])

print(f'num of selected feats: {len(features)}')

### 선별된 속성으로 학습 데이터 구성

In [ ]:
# features = list(df_selected.columns)

train_x = df_train[features]
test_x = df_val[features]
test_data = test_data[features]

test_data.info()

## 3. 모델 학습


### 모델 정의


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.ensemble import VotingClassifier

models = {}
models_name = {}
knns = [
    KNeighborsClassifier(n_neighbors=12, weights='distance', p=1),
    KNeighborsClassifier(n_neighbors=20, weights='distance', p=1),
    KNeighborsClassifier(n_neighbors=2, weights='distance', p=1),
    KNeighborsClassifier(n_neighbors=8, weights='distance', p=1),
    KNeighborsClassifier(n_neighbors=16, weights='distance', p=1)
]

# K최근접이웃
models['knn'] = KNeighborsClassifier(n_neighbors=12, weights='distance', p=1)
models_name['knn'] = 'K-Neighbors Classifier'

# 랜덤포레스트
models['rf'] = RandomForestClassifier(n_estimators=20, random_state=RANDOM_STATE, verbose=1, class_weight='balanced')
models_name['rf'] = 'Random Forest Classifier'

# AdaBoost Classifier
models['ada'] = AdaBoostClassifier()
models_name['ada'] = 'Ada Boost Classifier'

# Bagging Classifier
models['bag'] = BaggingClassifier()
models_name['bag'] = 'Bagging Classifier'

# Voting Classifier
models['vc'] = VotingClassifier(
    estimators=[
        ('KNN1', knns[0]),
        ('KNN2', knns[1]),
        ('KNN3', knns[2]),
        ('KNN4', knns[3]),
        ('KNN5', knns[4]),
    ],
    voting='soft',
    weights={0: 1, 1: 3}
)
models_name['vc'] = 'Voting Classifier'

for name in models.keys():
    print(f'* \'{models_name[name]}\': {models[name]}')

### 모델 학습


In [ ]:
total_start_time = time.time()

print_time()

In [ ]:
test_x.shape

In [ ]:
test_y

#### 랜덤 서치 모델 학습

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

# Define the hyperparameter grid for each model
param_grids = {
    'knn': {
        'n_neighbors': randint(10, 20),
        'weights': ['uniform', 'distance'],
        'p': randint(1, 2)
    },
    'rf': {
        'n_estimators': randint(10, 200),
        'max_depth': randint(3, 20),
        'min_samples_split': randint(2, 10),
        'min_samples_leaf': randint(1, 10),
        'bootstrap': [True, False],
        'random_state': [RANDOM_STATE]
    },
    # Add more grids for other models if necessary
}

# Perform RandomizedSearchCV for each model
for name in param_grids.keys():
    print(f'{models_name[name]} random search and training...\n')
    start = time.time()

    # Perform RandomizedSearchCV
    search = RandomizedSearchCV(models[name], param_grids[name], n_iter=20, scoring='f1', n_jobs=1, cv=5, random_state=RANDOM_STATE, verbose=1)
    search.fit(train_x, train_y)

    models[name] = search.best_estimator_
    print(f'Best hyperparameters for {models_name[name]}\n{search.best_params_}')

    end = time.time()
    print(f'{models_name[name]} trained with random search in {int(end - start) //  60} min {int(end - start) % 60} second\n')

#### 랜덤 서치 안 하는 모델 학습

안 하는 사유
1. 너무 오래걸림
2. 딱히 만질 하이퍼파라미터가 안보임(몰라서)

In [ ]:
models.keys()

In [ ]:
# 'ada': {
#     'estimator': [models['rf'], models['vc']],
#     'n_estimators': randint(50, 100).rvs(size=1),
#     'random_state': [RANDOM_STATE]
# },
# 'bag': {
#     'estimator': [models['knn'], models['rf'], models['ada'], models['vc']],
#     'n_estimators': randint(10, 50).rvs(size=1),
#     'random_state': [RANDOM_STATE]
# },

# AdaBoost Classifier
# knn이 sample_weight를 지원하지 않아 사용 불가
models['ada'] = AdaBoostClassifier(
    estimator=models['vc'],
    n_estimators=50,
    random_state=RANDOM_STATE
)

# Bagging Classifier
models['bag'] = BaggingClassifier(
    estimator=models['vc'],
    n_estimators=10,
    random_state=RANDOM_STATE
)

to_fit = ['vc', 'ada', 'bag']

for name in to_fit:
    print(f'{models_name[name]} training...')
    start = time.time()

    models[name].fit(train_x, train_y)

    end = time.time()
    print(f'{models_name[name]} trained with random search in {int(end - start) //  60} min {int(end - start) % 60} second\n')

In [ ]:
total_train_time = int(time.time() - total_start_time)

print(f'total train time: {total_train_time // 60} min {total_train_time % 60} second')

### 기록 확인

In [ ]:
# for name in best_models.keys():
#     models[name] = best_models[name]

for name in models.keys():
    print(f'* \'{models_name[name]}\': {models[name]}')

In [ ]:
preds = {}

for name in models.keys():
    preds[name] = models[name].predict(test_x)  # Use best_models instead of models
    
    prec = precision_score(test_y, preds[name], average=None)
    recall = recall_score(test_y, preds[name], average=None)
    f1score = f1_score(test_y, preds[name], average=None)
    
    print(f"{models_name[name]} 학습 결과\n***************************")
    print(f"- 정확도: {accuracy_score(test_y, preds[name]) :.4f}")
    print(f"- precision: {prec[1] :.4f}")
    print(f"- recall: {recall[1] :.4f}")
    print(f"- f1 score: {f1score[1] :.4f}")
    print(f'precision과 recall의 차이: {prec[1] - recall[1] :.4f}')
    print()

In [ ]:
for name in models.keys():
    confu_mat = confusion_matrix(test_y, preds[name], normalize='true')
    
    print(f"{models_name[name]} 학습 결과\n===========================")
    print(f"분류 결과:\n{classification_report(test_y, preds[name], target_names=['Normal', 'AbNormal'])}")
    print(f"혼동 행렬:\n{confu_mat}")
    sns.heatmap(confu_mat, annot=True, cmap='Blues', vmax = 1, vmin = 0)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title(f'{models_name[name]} confusion mat.')
    plt.show()
    print()

## 4. 제출하기


### 테스트 데이터 예측


In [ ]:
test_data.isna().sum().sum()

In [ ]:
df_test_x = test_data[features]

# 학습 데이터와 동일하게 정수화 안 함
# for col in df_test_x.columns:
#     try:
#         df_test_x.loc[:, col] = df_test_x[col].astype(int)
#     except:
#         continue

print_time()
df_test_x.info()

In [ ]:
models.keys()

In [ ]:
models_name.keys()

In [ ]:
test_preds = {}

for name in models.keys():
    test_preds[name] = models[name].predict(df_test_x)
    print(f'{models_name[name]} predicted {round(sum(test_preds[name]) / len(test_preds[name]) * 100, 2)}% AbNormal')
#     print()

In [ ]:
print_time()

# 제출용 모델 이름만 골라서 넣으면 됨
test_pred = test_preds['vc']

print(f'{round(sum(test_pred) / len(test_pred) * 100, 2)}% AbNormal')

test_pred

### 제출 파일 작성


In [ ]:
# 제출 데이터 읽어오기 (df_test는 전처리된 데이터가 저장됨)
# df_sub = pd.read_csv(ROOT_PREFIX + "submission.csv") # 에이머스 제출 시 주석 처리 필요
df_sub = pd.read_csv("submission.csv") # 구글 드라이브 실행 시 주석 처리 필요
df_sub["target"] = test_pred
df_sub.loc[df_sub['target'] >= 0.5, 'target'] = 1
df_sub.loc[df_sub['target'] < 0.5, 'target'] = 0

print(f"{sum(df_sub['target']) / len(df_sub['target']) * 100 :.2f}% AbNormal")

df_sub['target'] = df_sub['target'].astype(str)
df_sub.loc[df_sub['target'] == '1', 'target'] = 'AbNormal'
df_sub.loc[df_sub['target'] == '0', 'target'] = 'Normal'

print_time()

In [ ]:
print_time()

df_sub.head(10)

In [ ]:
# 제출 파일 저장
# df_sub.to_csv(ROOT_PREFIX + "submission.csv", index=False) # 에이머스 제출 시 주석 처리 필요
df_sub.to_csv("submission.csv", index=False) # 구글 코랩 실행 시 주석 처리 필요

print_time()

**우측 상단의 제출 버튼을 클릭해 결과를 확인하세요**
